# ChromaDB Vector Store – Visual Table

Load the project's ChromaDB knowledge base and display all stored chunks as a table.  
Run from **project root** or run the Setup cell first.  
Requires: `pandas` (`pip install pandas` if needed).

## Setup

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=True)

print("Project root:", ROOT)

## Load ChromaDB collection and fetch all data

In [ ]:
from vector_store.store import _get_collection

coll = _get_collection()
n = coll.count()
print(f"Collection: {coll.name}, total chunks: {n}")

if n == 0:
    print("No documents in the vector store. Upload PDFs via the app or 01_documents_api notebook.")
else:
    data = coll.get(include=["documents", "metadatas"])
    ids = data["ids"]
    docs = data.get("documents") or [""] * len(ids)
    metas = data.get("metadatas") or [{}] * len(ids)

## Build table (DataFrame)

In [ ]:
import pandas as pd

def truncate(s: str, max_len: int = 120) -> str:
    if not s or max_len <= 0:
        return ""
    s = str(s).strip().replace("\n", " ")
    return s[:max_len] + "..." if len(s) > max_len else s

rows = []
for i, doc_id in enumerate(ids):
    text = docs[i] if i < len(docs) else ""
    meta = metas[i] if i < len(metas) else {}
    rows.append({
        "#": i + 1,
        "id": doc_id,
        "document_name": meta.get("document_name", ""),
        "page": meta.get("page", "—"),
        "text_preview": truncate(text, 120),
        "text_length": len(text or ""),
    })

df = pd.DataFrame(rows)
df

## Display as styled table (wider text column)

In [ ]:
if len(df) > 0:
    pd.set_option("display.max_colwidth", 200)
    pd.set_option("display.width", None)
    display(df.style.set_properties(**{"text-align": "left"}).set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

## Summary by document

In [ ]:
if len(df) > 0:
    summary = df.groupby("document_name", as_index=False).agg(chunks=("#", "count"))
    # Add page range per document (when page is present)
    def pages_for_doc(name):
        pgs = df[df["document_name"] == name]["page"]
        pgs = [int(p) for p in pgs if p != "—" and isinstance(p, (int, float))]
        return sorted(set(pgs)) if pgs else "—"
    summary["pages"] = summary["document_name"].map(pages_for_doc)
    display(summary)